In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from PIL import Image
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity

print("="*60)
print("PCA KOMPRESI GAMBAR - KUALITAS TINGGI (TIDAK PECAH)")
print("="*60)

# 1. Upload gambar berwarna
print("\nUpload gambar berwarna (jpg/png):")
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

# 2. Simpan versi warna asli untuk output akhir
gambar_asli_warna = Image.open(file_path)

# 3. Konversi ke grayscale dan resize (200x200 agar detail terjaga)
UKURAN = (200, 200)  # ukuran lebih besar agar tidak pecah
gambar_gray = Image.open(file_path).convert("L")
gambar_gray_resize = gambar_gray.resize(UKURAN)
matriks_gambar = np.array(gambar_gray_resize, dtype=float)

h, w = matriks_gambar.shape
print(f"Ukuran matriks gambar: {h} x {w} = {h*w} pixel")

# 4. Exploratory Data Analysis (EDA) sederhana
plt.figure(figsize=(6,4))
plt.hist(matriks_gambar.ravel(), bins=50, color='gray', alpha=0.7)
plt.title("Histogram Intensitas Piksel (EDA)")
plt.xlabel("Intensitas (0-255)")
plt.ylabel("Frekuensi")
plt.show()

# 5. Centering (kurangi rata-rata kolom)
rata_kolom = np.mean(matriks_gambar, axis=0)
data_terpusat = matriks_gambar - rata_kolom

# 6. Matriks kovarians (ukuran w x w)
matriks_kovarians = np.cov(data_terpusat, rowvar=False)
print(f"Matriks kovarians shape: {matriks_kovarians.shape}")

# 7. Eigenvalue dan eigenvector
eigen_val, eigen_vec = np.linalg.eig(matriks_kovarians)
eigen_val = np.real(eigen_val)
eigen_vec = np.real(eigen_vec)

# 8. Urutkan descending
idx = np.argsort(eigen_val)[::-1]
eigen_val = eigen_val[idx]
eigen_vec = eigen_vec[:, idx]

# 9. Scree plot (titik elbow)
plt.figure(figsize=(8,5))
plt.plot(range(1, len(eigen_val)+1), eigen_val, 'bo-', linewidth=1)
plt.title("Scree Plot Eigenvalue (Cari Titik Elbow)")
plt.xlabel("Komponen PCA")
plt.ylabel("Eigenvalue")
plt.grid(True, alpha=0.3)
plt.show()

# 10. Bandingkan hasil untuk beberapa nilai k
nilai_k = [20, 50, 100, 150]
for k in nilai_k:
    if k > w:
        k = w
    eigen_vec_pilih = eigen_vec[:, :k]
    data_pca = np.dot(data_terpusat, eigen_vec_pilih)
    data_rekon = np.dot(data_pca, eigen_vec_pilih.T) + rata_kolom
    gambar_kompresi = np.clip(data_rekon, 0, 255).astype(np.uint8)

    # Evaluasi MSE, PSNR, SSIM
    mse = mean_squared_error(matriks_gambar, gambar_kompresi)
    psnr = peak_signal_noise_ratio(matriks_gambar, gambar_kompresi, data_range=255)
    ssim = structural_similarity(matriks_gambar, gambar_kompresi, data_range=255)

    print(f"\n=== Evaluasi untuk k = {k} komponen ===")
    print(f"MSE  : {mse:.2f}")
    print(f"PSNR : {psnr:.2f} dB")
    print(f"SSIM : {ssim:.4f}")

    # Tampilkan hasil side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(10,5))
    axes[0].imshow(matriks_gambar, cmap='gray')
    axes[0].set_title("Before (Grayscale Asli)")
    axes[0].axis("off")

    axes[1].imshow(gambar_kompresi, cmap='gray')
    axes[1].set_title(f"After PCA (k={k})")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

# 11. Tampilkan foto asli BERWARNA (di akhir)
gambar_warna_resize = gambar_asli_warna.resize(UKURAN)
plt.figure(figsize=(6,6))
plt.imshow(gambar_warna_resize)
plt.title("FOTO ASLI BERWARNA (HASIL AKHIR)")
plt.axis("off")
plt.show()

print("\n" + "="*60)
print("HASIL KOMPRESI DIEVALUASI DENGAN MSE, PSNR, SSIM, DAN RASIO KOMPRESI.")
print("Pilih nilai k yang seimbang: hemat data tapi tetap menjaga kualitas visual.")
print("="*60)

PCA KOMPRESI GAMBAR - KUALITAS TINGGI (TIDAK PECAH)

Upload gambar berwarna (jpg/png):
